In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import linregress
from scipy.optimize import curve_fit

# =====================================================
# EINSTELLUNGEN
# =====================================================

filename = "kinetics_empty.csv"

# =====================================================
# DATEN EINLESEN
# =====================================================

data = pd.read_csv(filename, sep=";")

# Temperatur in Kelvin
data["Temperature_K"] = data["Temperature_C"] + 273.15

temps = np.sort(data["Temperature_K"].unique())

# =====================================================
# k(T) BESTIMMEN
# =====================================================

k_values = []
k_errors = []
T_values = []

print("\n===================================")
print("FIT JE TEMPERATUR")
print("===================================\n")

for T in temps:

    subset = data[data["Temperature_K"] == T].copy()

    tau = subset["ResidenceTime_s"].values
    signal = subset["Signal"].values

    # Sicherheit: sortieren
    idx = np.argsort(tau)
    tau = tau[idx]
    signal = signal[idx]

    # Nur positive Werte
    valid = signal > 0

    tau = tau[valid]
    signal = signal[valid]

    if len(tau) < 3:
        print(f"{T:.1f} K übersprungen (<3 Punkte)")
        continue

    # ===================================
    # ln(I/I0) = -k*t
    # ===================================

    y = np.log(signal)

    slope, intercept, r_value, p_value, stderr = linregress(
        tau,
        y
    )

    k = -slope

    k_values.append(k)
    k_errors.append(stderr)
    T_values.append(T)

    print(
        f"T = {T:.1f} K | "
        f"k = {k:.4e} s^-1 | "
        f"R² = {r_value**2:.4f}"
    )

    # -----------------------------------
    # Kontrollplot
    # -----------------------------------

    plt.figure(figsize=(5,4))

    plt.scatter(
        tau,
        y,
        label="Data"
    )

    plt.plot(
        tau,
        intercept + slope*tau,
        label=f"k = {k:.3e} s$^{{-1}}$"
    )

    plt.xlabel("Residence time / s")
    plt.ylabel("ln(I/I$_0$)")
    plt.title(f"{T:.0f} K")
    plt.legend()

    plt.tight_layout()
    plt.savefig(f"Fit_{int(T)}K.png", dpi=300)

# =====================================================
# ARRHENIUS FIT
# =====================================================

k_values = np.array(k_values)
k_errors = np.array(k_errors)
T_values = np.array(T_values)

R = 8.314


def arrhenius(T, A, Ea):
    return A * np.exp(-Ea/(R*T))


popt, pcov = curve_fit(
    arrhenius,
    T_values,
    k_values,
    p0=[1e5, 5e4]
)

A_fit, Ea_fit = popt

sigma = np.sqrt(np.diag(pcov))

print("\n===================================")
print("ARRHENIUS FIT")
print("===================================\n")

print(f"A  = {A_fit:.3e} s^-1")
print(f"Ea = {Ea_fit/1000:.2f} kJ/mol")

print("\nUnsicherheiten:")

print(f"σ(A)  = {sigma[0]:.3e}")
print(f"σ(Ea) = {sigma[1]/1000:.2f} kJ/mol")

# =====================================================
# ARRHENIUS PLOT
# =====================================================

plt.figure(figsize=(6,5))

plt.scatter(
    1000/T_values,
    np.log(k_values),
    s=60,
    label="Experimental"
)

T_fit = np.linspace(
    min(T_values),
    max(T_values),
    200
)

plt.plot(
    1000/T_fit,
    np.log(arrhenius(T_fit, *popt)),
    label="Arrhenius fit"
)

plt.xlabel("1000/T (K$^{-1}$)")
plt.ylabel("ln(k)")
plt.legend()

plt.tight_layout()
plt.savefig("Arrhenius.png", dpi=300)

# =====================================================
# k(T) SPEICHERN
# =====================================================

results = pd.DataFrame({
    "Temperature_K": T_values,
    "k_tot_s-1": k_values,
    "k_error": k_errors
})

results.to_csv(
    "k_tot_results.csv",
    sep=";",
    index=False
)

print("\nDatei gespeichert:")
print("k_tot_results.csv")

plt.show()